In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from pydantic import BaseModel
from IPython.display import Image
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import uuid

In [16]:
from dotenv import load_dotenv
load_dotenv("../LangChainModels/.env")

True

In [17]:
llm = ChatOpenAI()

In [18]:
class JokeState(TypedDict):
    topic : str
    joke : str 
    explaination : str 

In [19]:
def generate_joke(state: JokeState)-> JokeState:
    topic = state['topic']
    prompt = f"generate a joke on this topic{topic}"
    response = llm.invoke(prompt)
    return {
        'joke': response.content
    }

In [20]:
def generate_explaination(state: JokeState ) -> JokeState:
    prompt = f"write a short explaination for the following joke : {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explaination': response}


In [21]:
graph = StateGraph(JokeState)

In [22]:
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination )

In [23]:
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

In [ ]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [26]:
thread_id = str(uuid.uuid4)
config1 = {'configurable' : {'thread_id' : thread_id}}
result = app.invoke({'topic': 'pizza'}, config=config1)
print(result)

{'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!', 'explaination': 'This joke plays on the double meaning of "crusty," which can refer to both someone feeling sick or unwell, and also to the texture of the crust on a pizza. In this case, the pizza went to the doctor because it was feeling unwell, or "crusty."'}


In [ ]:
app.get_state(config1) #final state values

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!', 'explaination': 'This joke plays on the double meaning of "crusty," which can refer to both someone feeling sick or unwell, and also to the texture of the crust on a pizza. In this case, the pizza went to the doctor because it was feeling unwell, or "crusty."'}, next=(), config={'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>', 'checkpoint_ns': '', 'checkpoint_id': '1f1c0d36-0130-6d40-8002-683baf5860dd'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-05T15:42:30.768941+00:00', parent_config={'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>', 'checkpoint_ns': '', 'checkpoint_id': '1f1c0d35-f384-6bdf-8001-c1f8ac2e3666'}}, tasks=(), interrupts=())

In [42]:
list(app.get_state_history(config1)) #intermediate state values

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a pizza the action!', 'explaination': 'This joke is a pun on the phrase "get a piece of the action," which means to participate or be involved in something exciting or interesting. In this context, the pizza went to the party because it wanted to be a part of the fun and excitement.'}, next=(), config={'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>', 'checkpoint_ns': '', 'checkpoint_id': '1f1c0d94-5e38-6798-8003-9b940af595e5'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-05T16:24:43.817116+00:00', parent_config={'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>', 'checkpoint_ns': '', 'checkpoint_id': '1f1c0d94-5435-6efd-8002-a4f96f056f28'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a pizza the action!'}, 

# Time Travel

In [39]:
app.get_state({'configurable': {'checkpoint_id': '1f1c0d35-d2e1-6824-8000-80022dd77b48','thread_id': '<function uuid4 at 0x00000207423C1D00>'}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'checkpoint_id': '1f1c0d35-d2e1-6824-8000-80022dd77b48', 'thread_id': '<function uuid4 at 0x00000207423C1D00>'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-05T15:42:25.912982+00:00', parent_config={'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>', 'checkpoint_ns': '', 'checkpoint_id': '1f1c0d35-d2c9-6095-bfff-da8307d22199'}}, tasks=(PregelTask(id='865368c3-9a8e-0f2f-ebfb-80f3a7884baa', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crusty!'}),), interrupts=())

In [41]:
app.invoke(None, {'configurable': {'checkpoint_id': '1f1c0d35-d2e1-6824-8000-80022dd77b48','thread_id': '<function uuid4 at 0x00000207423C1D00>'}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the party?\n\nBecause it wanted to get a pizza the action!',
 'explaination': 'This joke is a pun on the phrase "get a piece of the action," which means to participate or be involved in something exciting or interesting. In this context, the pizza went to the party because it wanted to be a part of the fun and excitement.'}

## Updating State

In [45]:
config = {
    'configurable': {
        'thread_id': '<function uuid4 at 0x00000207423C1D00>',
        'checkpoint_ns': '',
        'checkpoint_id': '1f1c0d94-5e38-6798-8003-9b940af595e5'
    }
}

In [46]:
app.update_state(
    config,
    {"topic": "samosa"}
)

{'configurable': {'thread_id': '<function uuid4 at 0x00000207423C1D00>',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1c0dab-4a14-609a-8004-45eec84fb196'}}

In [48]:
app.invoke(
    {"topic": "samosa"},
    config
)

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school?\n\nTo become a little more well-rounded!',
 'explaination': 'This joke plays on the idea of a samosa, a popular Indian snack, being "well-rounded" in terms of its shape and also "well-rounded" in terms of being well-educated or knowledgeable by going to school. It\'s a play on words that adds humor to the punchline.'}